# REE Extraction Basics: Using the difflow_ree Module

This notebook demonstrates the **difflow_ree** plugin for rare earth element (REE) solvent extraction.

## Background

The difflow_ree module provides:
- **Database** of 15 REE elements with properties
- **5 extractant systems** (D2EHPA, PC88A, Cyanex272, TBP, naphthenic acid)
- **pH-dependent distribution models**
- **Unit operations** for extraction, scrubbing, stripping
- **Pre-built flowsheet templates**

## What You'll Learn

1. Access REE element and extractant databases
2. Calculate pH-dependent distribution coefficients
3. Simulate multi-stage extraction units
4. Perform sensitivity analysis and optimization
5. Use automatic differentiation for gradients
6. Read the provenance of every number before quoting it

## A warning before any number in here is copied

Three of the five extractant records are `HAND_TUNED` --- chosen so example
code behaves, with no external basis. Only PC88A's pH coefficients are fit to
measured data (Torres et al. 2021, refit in #270), and only five of the fifteen
element prices carry a citation (USGS Mineral Commodity Summaries 2026).
Section 9 shows how to ask any field where it came from. Everything in this
notebook is fine for exercising the solver and the gradients; none of it is
fine behind a published number.

In [1]:
import jax
import jax.numpy as jnp
from jax import grad

# Enable 64-bit precision for numerical stability
jax.config.update("jax_enable_x64", True)

# Import difflow_ree components
from difflow_ree import (
    # Database access
    get_element,
    get_extractant,
    list_ree_elements,
    list_extractants,
    # Distribution model
    REEDistribution,
    # Unit operations
    REEExtractor,
    REEExtractorParams,
)

# Provenance: every database field can say where it came from
from difflow_ree.provenance import explain

# Import stream utilities
from difflow.streams import make_stream, get_flows

## 1. Exploring the REE Database

The difflow_ree module includes a comprehensive database of REE properties.

In [2]:
# List all available REE elements
elements = list_ree_elements()
print(f"Available REE elements ({len(elements)}):")
print("  " + ", ".join(elements))

# Get detailed properties for specific elements
print("\nElement Properties:")
print("="*78)
print(f"{'Symbol':<8} {'Name':<15} {'Group':<10} {'Price ($/kg)':<15} {'Price source':<15}")
print("-"*78)

for symbol in ["La", "Nd", "Eu", "Dy"]:
    elem = get_element(symbol)
    prov = explain("elements", f"elements.{symbol}.price_usd_kg")
    print(f"{elem.symbol:<8} {elem.name:<15} {elem.group:<10} "
          f"${elem.price_usd_kg:<14.2f} {prov.cls + ' / ' + prov.source:<15}")

# Prices are oxide (REO) basis, 2025 annual averages where sourced. Five of the
# fifteen elements have a citation; Dy, Tb and the rest do not, and the numbers
# beside them are estimates. Do not read the dated citation on La as saying
# anything about Dy.


Available REE elements (15):
  La, Ce, Pr, Nd, Sm, Eu, Gd, Tb, Dy, Y, Ho, Er, Tm, Yb, Lu

Element Properties:
Symbol   Name            Group      Price ($/kg)    Price source   
------------------------------------------------------------------------------
La       Lanthanum       light      $1.00           REFERENCE / USGS26
Nd       Neodymium       light      $69.00          REFERENCE / USGS26
Eu       Europium        middle     $27.00          REFERENCE / USGS26
Dy       Dysprosium      heavy      $450.00         ESTIMATED / EST


## 2. Exploring Extractant Database

Industrial extractants for REE separation.

In [3]:
# List available extractants
extractants = list_extractants()
print(f"Available extractants ({len(extractants)}):")
for ext_name in extractants:
    print(f"  • {ext_name}")

# Get D2EHPA properties
d2ehpa = get_extractant("D2EHPA")
print(f"\nD2EHPA Properties:")
print(f"  Full name: {d2ehpa.full_name}")
print(f"  Formula: {d2ehpa.formula}")
print(f"  MW: {d2ehpa.molecular_weight:.2f} g/mol")
print(f"  pKa: {d2ehpa.pKa}")
print(f"  Type: {d2ehpa.extractant_type}")
print(f"  Typical concentration: {d2ehpa.typical_concentration} M")
print(f"  Valid pH range: {d2ehpa.valid_ph_range}")
print(f"  Cost: ${d2ehpa.cost_usd_kg}/kg")

Available extractants (5):
  • D2EHPA
  • PC88A
  • Cyanex272
  • TBP
  • naphthenic_acid

D2EHPA Properties:
  Full name: Di-2-ethylhexyl phosphoric acid
  Formula: (C8H17O)2PO2H
  MW: 322.43 g/mol
  pKa: 3.24
  Type: acidic_phosphoric
  Typical concentration: 0.5 M
  Valid pH range: (1.0, 5.0)
  Cost: $8.0/kg


## 3. Distribution Coefficient Calculations

Distribution coefficients (D) determine how REEs partition between aqueous and organic phases:

$$D = \frac{[\text{REE}]_{\text{organic}}}{[\text{REE}]_{\text{aqueous}}}$$

The model includes:
- pH dependence: $\log_{10}(D) = a + b \cdot pH + c \cdot pH^2$
- Temperature correction
- Extractant concentration effects

In [4]:
# Create distribution model for D2EHPA
dist = REEDistribution(
    extractant="D2EHPA",
    elements=("La", "Nd", "Dy"),
    concentration=0.5,  # 0.5 M in organic phase
)

# Calculate D values at pH 3.0
D_values = dist.get_D_all(pH=3.0, T=298.15)

print("Distribution Coefficients at pH 3.0, 25°C:")
print("="*50)
for elem, D in D_values.items():
    print(f"  D({elem}) = {float(D):8.4f}")

# Calculate separation factors
SF_Nd_La = float(D_values["Nd"] / D_values["La"])
SF_Dy_Nd = float(D_values["Dy"] / D_values["Nd"])

print(f"\nSeparation Factors:")
print(f"  SF(Nd/La) = {SF_Nd_La:.2f}")
print(f"  SF(Dy/Nd) = {SF_Dy_Nd:.2f}")
print("\n💡 Higher SF = easier separation")

Distribution Coefficients at pH 3.0, 25°C:
  D(La) =   0.0309
  D(Nd) =   0.5495
  D(Dy) =  51.2861

Separation Factors:
  SF(Nd/La) = 17.78
  SF(Dy/Nd) = 93.33

💡 Higher SF = easier separation


## 4. pH Effect on Distribution

D values increase exponentially with pH for acidic extractants.

In [5]:
# Effect of pH on distribution coefficients
pH_values = [2.0, 2.5, 3.0, 3.5, 4.0]

print("pH Effect on Distribution Coefficients (D2EHPA, 0.5 M):")
print("="*60)
print(f"{'pH':<6} {'D(La)':<12} {'D(Nd)':<12} {'D(Dy)':<12}")
print("-"*60)

for pH in pH_values:
    D_vals = dist.get_D_all(pH=pH, T=298.15)
    print(f"{pH:<6.1f} {float(D_vals['La']):<12.4f} {float(D_vals['Nd']):<12.4f} {float(D_vals['Dy']):<12.4f}")

# How steep is that, really? Read it off the table rather than asserting it.
print("\n📊 D rises by 10**b per pH unit, where b is the number of protons the")
print("   exchange releases. Measured across the rows above:")
for lo, hi in [(2.0, 3.0), (3.0, 4.0)]:
    for elem in ["La", "Nd", "Dy"]:
        ratio = float(dist.get_D(elem, hi) / dist.get_D(elem, lo))
        print(f"     D({elem}) x {ratio:8.1f}  from pH {lo} to {hi}")

print("""
   Cation exchange on a dimeric acidic extractant,

       RE(3+) + 3 (HA)2  <->  RE(HA2)3 + 3 H+,

   releases three protons, so mass action demands b = 3 exactly -- a factor of
   1000 per pH unit, not 10. This record gives ~300x because D2EHPA's
   coefficients are HAND_TUNED with b = 2.45, below the stoichiometric slope.
   PC88A's coefficients, refit to Torres et al. (2021) in #270, are pinned at
   b = 3 for every element, which is what makes its separation factors exactly
   pH-independent (section 8).""")


pH Effect on Distribution Coefficients (D2EHPA, 0.5 M):
pH     D(La)        D(Nd)        D(Dy)       
------------------------------------------------------------
2.0    0.0001       0.0017       0.0724      
2.5    0.0021       0.0307       1.9165      
3.0    0.0309       0.5495       51.2861     
3.5    0.4704       9.9426       1388.3533   
4.0    7.2444       181.9701     38018.9396  

📊 D rises by 10**b per pH unit, where b is the number of protons the
   exchange releases. Measured across the rows above:
     D(La) x    223.9  from pH 2.0 to 3.0
     D(Nd) x    316.2  from pH 2.0 to 3.0
     D(Dy) x    707.9  from pH 2.0 to 3.0
     D(La) x    234.4  from pH 3.0 to 4.0
     D(Nd) x    331.1  from pH 3.0 to 4.0
     D(Dy) x    741.3  from pH 3.0 to 4.0

   Cation exchange on a dimeric acidic extractant,

       RE(3+) + 3 (HA)2  <->  RE(HA2)3 + 3 H+,

   releases three protons, so mass action demands b = 3 exactly -- a factor of
   1000 per pH unit, not 10. This record gives ~300

## 5. Multi-Stage Extraction Unit

Simulate a counter-current extraction cascade.

The organic solvent stream names its carriers explicitly — the extractant (D2EHPA) and the diluent (kerosene) — rather than a generic `"Organic"` species. The extractant molar flow is what sets the loading capacity of the organic phase (#191), and a solvent that names neither carrier is now rejected instead of being silently assigned an organic flow of 1.0 (#192).


In [6]:
# Create extraction unit parameters
params = REEExtractorParams(
    n_stages=5,
    extractant="D2EHPA",
    elements=("La", "Nd", "Dy"),
    pH=3.0,
)

# Create extractor
extractor = REEExtractor(params)

# Define feed streams
feed = make_stream(
    flows={
        "H2O": 10.0,
        "La": 0.01,
        "Nd": 0.02,
        "Dy": 0.01,
    },
    T=298.15,
    P=101325.0,
)

# Organic solvent stream.
#
# The stream must name the *extractant* and the *diluent* as species. A
# solvent whose carrier matches neither now raises instead of silently
# defaulting the organic flow to 1.0 (#192), and when loading is enabled the
# extractant molar flow is what sets the capacity of the organic phase
# (capacity = F_extractant / m, #191).
#
# 0.5 M D2EHPA in kerosene is roughly 10 mol% extractant (kerosene is
# ~0.75 g/mL and ~170 g/mol, so ~4.4 mol/L of diluent against 0.5 mol/L of
# extractant). The total organic flow is unchanged at 8.0 mol/s; it is just
# split 0.8 D2EHPA / 7.2 kerosene.
solvent = make_stream(
    flows={
        "D2EHPA": 0.8,
        "kerosene": 7.2,
        "La": 0.0,
        "Nd": 0.0,
        "Dy": 0.0,
    },
    T=298.15,
    P=101325.0,
)

# Run extraction
raffinate, extract, info = extractor(feed, solvent)

# Analyze results
feed_flows = get_flows(feed)
raff_flows = get_flows(raffinate)
ext_flows = get_flows(extract)

print("Extraction Results (5 stages, pH 3.0):")
print("="*70)
print(f"{'Element':<10} {'Feed':<12} {'Raffinate':<12} {'Extract':<12} {'Recovery %':<12}")
print("-"*70)

for elem in ["La", "Nd", "Dy"]:
    feed_val = float(feed_flows[elem])
    raff_val = float(raff_flows[elem])
    ext_val = float(ext_flows[elem])
    recovery = (ext_val / feed_val) * 100
    
    print(f"{elem:<10} {feed_val:<12.4f} {raff_val:<12.4f} {ext_val:<12.4f} {recovery:<12.1f}")

# Calculate purities
total_REE_extract = sum(float(ext_flows[e]) for e in ["La", "Nd", "Dy"])
nd_purity = float(ext_flows["Nd"]) / total_REE_extract * 100

print(f"\nExtract Composition:")
print(f"  Nd purity (among REEs): {nd_purity:.1f}%")

Extraction Results (5 stages, pH 3.0):
Element    Feed         Raffinate    Extract      Recovery %  
----------------------------------------------------------------------
La         0.0100       0.0098       0.0002       2.5         
Nd         0.0200       0.0113       0.0087       43.4        
Dy         0.0100       0.0000       0.0100       100.0       

Extract Composition:
  Nd purity (among REEs): 45.9%


## 6. Automatic Differentiation: Gradients

Compute exact gradients for sensitivity analysis and optimization.

In [7]:
def nd_recovery(pH):
    """Nd recovery vs pH, by the same equation the REEExtractor solves."""
    dist_pH = REEDistribution(
        extractant="D2EHPA",
        elements=("La", "Nd", "Dy"),
        concentration=0.5,
    )

    D_val = dist_pH.get_D("Nd", pH, T=298.15)

    S_F = 0.8   # solvent / feed ratio, as in section 5
    N = 5.0     # stages, as in section 5

    # Counter-current cascade -- Kremser:
    #
    #     fraction extracted = (E**(N+1) - E) / (E**(N+1) - 1),   E = D * S/F
    #
    # This cell used to compute 1 - 1/(1 + E)**N instead. That is the
    # CROSS-CURRENT result -- N stages each contacted with FRESH solvent -- and
    # it is not a small difference: at pH 3.0 it reports 83.8% where the
    # five-stage cascade of section 5 delivers 43.4%. Counter-current reuses
    # one solvent stream, so every stage after the first meets organic that is
    # already partly loaded.
    #
    # E = 1 is a removable singularity (the limit is N/(N+1)). This cell stays
    # well away from it; difflow_ree guards it with a jnp.where.
    E = D_val * S_F
    E_Np1 = E ** (N + 1.0)
    recovery = (E_Np1 - E) / (E_Np1 - 1.0)

    return recovery

# Compute gradient
pH_operating = 3.0
dR_dpH = grad(nd_recovery)(pH_operating)
R = float(nd_recovery(pH_operating))

print("Sensitivity Analysis:")
print("="*50)
print(f"Operating pH: {pH_operating}")
print(f"Extraction factor E = D(Nd)*S/F: {float(dist.get_D('Nd', pH_operating)) * 0.8:.4f}")
print(f"Nd recovery: {R*100:.2f}%   (section 5's cascade: 43.4%)")
print(f"\n\u2202(Nd recovery)/\u2202pH = {float(dR_dpH):.4f} per pH unit")
print(f"\n\U0001f4a1 Interpretation:")
print(f"  \u2022 The derivative is local. 0.1 pH units up is worth about")
print(f"    {float(dR_dpH)*0.1*100:.1f} points of recovery here, but the curve is an S,")
print(f"    so do not extrapolate it a whole pH unit -- check by evaluating:")
for pH in [2.9, 3.0, 3.1, 3.5]:
    print(f"      pH {pH:.1f}  recovery {float(nd_recovery(pH))*100:6.2f}%")
print(f"  \u2022 pH control is critical: E passes through 1 near pH 3.1, which is")
print(f"    exactly where the cascade is most sensitive to it.")


Sensitivity Analysis:
Operating pH: 3.0
Extraction factor E = D(Nd)*S/F: 0.4396
Nd recovery: 43.56%   (section 5's cascade: 43.4%)

∂(Nd recovery)/∂pH = 2.4170 per pH unit

💡 Interpretation:
  • The derivative is local. 0.1 pH units up is worth about
    24.2 points of recovery here, but the curve is an S,
    so do not extrapolate it a whole pH unit -- check by evaluating:
      pH 2.9  recovery  24.65%
      pH 3.0  recovery  43.56%
      pH 3.1  recovery  71.85%
      pH 3.5  recovery 100.00%
  • pH control is critical: E passes through 1 near pH 3.1, which is
    exactly where the cascade is most sensitive to it.


## 7. Optimization: Find Optimal pH

Use gradient descent to maximize Nd purity.

In [8]:
def nd_purity_objective(pH):
    """Nd purity among all REEs in a single equilibrium contact."""
    dist_pH = REEDistribution(
        extractant="D2EHPA",
        elements=("La", "Nd", "Dy"),
        concentration=0.5,
    )

    D_vals = dist_pH.get_D_all(pH, T=298.15)

    # Single equilibrium stage: in the extract, [REE] is proportional to
    # D * [REE]_feed.
    feed_ratio = {"La": 0.01, "Nd": 0.02, "Dy": 0.01}

    extract_amounts = {
        elem: D_vals[elem] * feed_ratio[elem]
        for elem in ["La", "Nd", "Dy"]
    }

    total = sum(extract_amounts.values())
    purity = extract_amounts["Nd"] / (total + 1e-10)

    return purity

# Look at the objective before optimizing it. b(Dy) > b(Nd) > b(La) on this
# record, so raising pH pulls Dy into the organic faster than Nd: Nd's share of
# the extract falls monotonically. This objective has a BOUNDARY solution.
pH_lo, pH_hi = get_extractant("D2EHPA").valid_ph_range

print("Objective over D2EHPA's own validity window:")
print("="*66)
print(f"{'pH':<8} {'Nd purity %':<14} {'d(purity)/dpH':<16} {'Nd recovery %':<14}")
print("-"*66)
for pH in [1.0, 2.0, 3.0, 4.0, 5.0]:
    print(f"{pH:<8.1f} {float(nd_purity_objective(pH))*100:<14.2f} "
          f"{float(grad(nd_purity_objective)(pH)):<16.5f} "
          f"{float(nd_recovery(pH))*100:<14.2f}")

# Projected gradient ascent. The step has to be scaled to the gradient, which
# is O(0.05) here -- a learning rate of 0.1 moves pH by 0.005 an iteration and
# 20 of those go nowhere, which is what this cell used to report as an optimum.
pH_opt = 3.0
learning_rate = 20.0

print("\nProjected gradient ascent:")
print("="*50)
print(f"{'Iter':<8} {'pH':<10} {'Nd Purity %':<15}")
print("-"*50)

for i in range(50):
    gradient = grad(nd_purity_objective)(pH_opt)
    pH_opt = jnp.clip(pH_opt + learning_rate * gradient, pH_lo, pH_hi)

    if (i + 1) % 10 == 0:
        purity = nd_purity_objective(pH_opt)
        print(f"{i+1:<8} {float(pH_opt):<10.3f} {float(purity)*100:<15.2f}")

final_purity = nd_purity_objective(pH_opt)
final_grad = float(grad(nd_purity_objective)(pH_opt))
at_bound = bool(abs(float(pH_opt) - pH_lo) < 1e-9)

print(f"\nConverged pH: {float(pH_opt):.3f}   (window [{pH_lo}, {pH_hi}])")
print(f"  Nd purity:   {float(final_purity)*100:.2f}%")
print(f"  gradient at the solution: {final_grad:+.5f} per pH unit")
print(f"  at the lower bound: {at_bound}")

print("""
\u26a0\ufe0f  Read this result before believing it.

The optimum is the LOW-pH CORNER of the window, and the gradient there is still
negative -- the optimizer is not stationary, it is pinned. That is the correct
answer to the question asked, and the question is the wrong one: purity alone
is maximized where almost nothing is extracted. The recovery column above shows
it -- at pH 1.0 the single-stage Nd purity is the best available and the
five-stage cascade recovers essentially no Nd at all.

Two honest ways forward, neither of which is "run more iterations":
  * optimize purity SUBJECT TO a recovery floor, or maximize the product; and
  * separate La/Nd from Dy in one contactor and Nd from La in another. No
    single pH separates three elements whose D values are monotone in pH --
    that is what a cascade with scrub and strip sections is for (notebook 04).
""")


Objective over D2EHPA's own validity window:
pH       Nd purity %    d(purity)/dpH    Nd recovery % 
------------------------------------------------------------------
1.0      9.65           -0.06958         0.00          
2.0      4.57           -0.03505         0.14          
3.0      2.10           -0.01653         43.56         
4.0      0.95           -0.00757         100.00        
5.0      0.43           -0.00342         100.00        

Projected gradient ascent:
Iter     pH         Nd Purity %    
--------------------------------------------------


10       1.000      9.65           
20       1.000      9.65           
30       1.000      9.65           
40       1.000      9.65           


50       1.000      9.65           

Converged pH: 1.000   (window [1.0, 5.0])
  Nd purity:   9.65%
  gradient at the solution: -0.06958 per pH unit
  at the lower bound: True

⚠️  Read this result before believing it.

The optimum is the LOW-pH CORNER of the window, and the gradient there is still
negative -- the optimizer is not stationary, it is pinned. That is the correct
answer to the question asked, and the question is the wrong one: purity alone
is maximized where almost nothing is extracted. The recovery column above shows
it -- at pH 1.0 the single-stage Nd purity is the best available and the
five-stage cascade recovers essentially no Nd at all.

Two honest ways forward, neither of which is "run more iterations":
  * optimize purity SUBJECT TO a recovery floor, or maximize the product; and
  * separate La/Nd from Dy in one contactor and Nd from La in another. No
    single pH separates three elements whose D values are monotone in pH --
    that is what a cascade with scrub a

## 8. Comparing Different Extractants

Compare D2EHPA, PC88A and Cyanex272 for Nd extraction --- each inside its own
validity window, because they no longer share one.


In [9]:
extractant_list = ["D2EHPA", "PC88A", "Cyanex272"]

# The three acidic extractants have no pH in common. Since the #270 refit
# PC88A's window is [0.1, 2.5] -- the span Torres et al. (2021) actually
# measured -- while D2EHPA's is [1.0, 5.0] and Cyanex 272's is [3.0, 7.0].
# This cell used to compare all three at pH 3.0, which extrapolates PC88A four
# and a half decades past the end of its data (b = 3).
#
# Compare each at its OWN equal-split pH instead: the pH where D(Nd) = 1, where
# a 1:1 cascade puts half the Nd in each phase. That is the operating point the
# reagent is actually chosen around, and it is inside every window by
# construction.

def equal_split_pH(dist, lo, hi, elem="Nd"):
    """Bisect for the pH in [lo, hi] where D(elem) = 1."""
    f = lambda p: float(jnp.log10(dist.get_D(elem, p, T=298.15)))
    if f(lo) * f(hi) > 0:
        return None
    for _ in range(60):
        mid = 0.5 * (lo + hi)
        if f(lo) * f(mid) <= 0:
            hi = mid
        else:
            lo = mid
    return 0.5 * (lo + hi)

print("Each extractant at its own equal-split pH (D(Nd) = 1):")
print("="*86)
print(f"{'Extractant':<12} {'window':<14} {'conc M':<8} {'pH':<7} "
      f"{'SF(Nd/La)':<11} {'SF(Dy/Nd)':<11} {'coefficients':<12}")
print("-"*86)

for ext_name in extractant_list:
    ext = get_extractant(ext_name)
    lo, hi = ext.valid_ph_range
    dist_compare = REEDistribution(
        extractant=ext_name,
        elements=("La", "Nd", "Dy"),
        concentration=ext.typical_concentration,
    )

    pH_eq = equal_split_pH(dist_compare, lo, hi)
    D_vals = dist_compare.get_D_all(pH=pH_eq, T=298.15)
    prov = explain("extractants", f"extractants.{ext_name}.ph_coefficients.Nd.a")

    print(f"{ext_name:<12} [{lo}, {hi}]".ljust(27)
          + f"{ext.typical_concentration:<8.2f} {pH_eq:<7.3f} "
            f"{float(D_vals['Nd']/D_vals['La']):<11.1f} "
            f"{float(D_vals['Dy']/D_vals['Nd']):<11.1f} {prov.cls:<12}")

print("""
\U0001f4a1 What this table does and does not say:

  * On these records D2EHPA has the BETTER light-REE selectivity, not PC88A.
    Earlier versions of this notebook asserted the opposite. PC88A's real
    industrial advantage over D2EHPA is not selectivity, it is stripping: the
    weaker acid (pKa 4.10 vs 3.24) gives back its loaded metal at far lower
    acid strength, which is why it displaced D2EHPA for the middle and heavy
    separations (Nash 1993).

  * All three separation factors for PC88A are pH-, temperature- and
    concentration-INDEPENDENT: since #270 every element on that record shares
    one slope b = 3, so log10(beta) = a_i - a_j exactly. Recompute the row at
    any pH in its window and it will not move.

  * The equal-split pH column is ORDERED BACKWARDS from the acid strengths.
    The stronger acid, D2EHPA, should extract at the LOWER pH, and it does not
    here. Only PC88A's coefficients are MEASURED; D2EHPA's and Cyanex 272's are
    HAND_TUNED, as the last column says. The discrepancy is a statement about
    the two hand-tuned records, not about the measurement. Do not compare
    absolute D values across records of different provenance.

  * Cyanex 272 does have the largest Dy/Nd factor, and it is the reagent of
    choice where a heavy/middle cut is what you need.""")


Each extractant at its own equal-split pH (D(Nd) = 1):
Extractant   window         conc M   pH      SF(Nd/La)   SF(Dy/Nd)   coefficients
--------------------------------------------------------------------------------------
D2EHPA       [1.0, 5.0]    0.50     3.104   18.4        101.4       HAND_TUNED  
PC88A        [0.1, 2.5]    0.50     1.026   10.0        486.5       MEASURED    
Cyanex272    [3.0, 7.0]    0.30     3.736   32.4        1017.9      HAND_TUNED  

💡 What this table does and does not say:

  * On these records D2EHPA has the BETTER light-REE selectivity, not PC88A.
    Earlier versions of this notebook asserted the opposite. PC88A's real
    industrial advantage over D2EHPA is not selectivity, it is stripping: the
    weaker acid (pKa 4.10 vs 3.24) gives back its loaded metal at far lower
    acid strength, which is why it displaced D2EHPA for the middle and heavy
    separations (Nash 1993).

  * All three separation factors for PC88A are pH-, temperature- and
    conce

## 9. Where Did These Numbers Come From?

Every field in the database carries a provenance class. Ask before you quote.


In [10]:
from difflow_ree.provenance import coverage

# The whole-database picture: how many fields of each class.
print("Provenance coverage across difflow_ree:")
for cls, n in sorted(coverage().items(), key=lambda kv: -kv[1]):
    print(f"  {cls:<12} {n:>4}")

# And any single field, in full.
print()
print(explain("extractants", "extractants.D2EHPA.ph_coefficients.Nd.b"))


Provenance coverage across difflow_ree:
  REFERENCE     186
  CONVENTION    156
  HAND_TUNED    123
  MEASURED       94
  DERIVED        32
  ESTIMATED      18
  CONSTRUCTED     6

extractants:extractants.D2EHPA.ph_coefficients.Nd.b = 2.45
  source   HAND_TUNED  [HAND_TUNED]
  citation Chosen so that example code produces reasonable behaviour.
  about HAND_TUNED: *** NO EXTERNAL BASIS OF ANY KIND. *** These numbers were selected to make
           demonstrations converge and plots look sensible. They predate any
           provenance discipline in this package. They are fine for exercising
           the solver, testing gradients and teaching the API. They must never
           appear in, or behind, a published number. Run
           audit(cls="HAND_TUNED") to see the full list.
  *** HAND_TUNED: do not put a published number behind this.


## Summary

This notebook demonstrated:

1. **Database access** --- REE properties and extractant data
2. **Distribution models** --- pH-dependent D values, and how steep they really are
3. **Multi-stage extraction** --- counter-current cascades, by Kremser
4. **Automatic differentiation** --- exact gradients for sensitivity
5. **Optimization** --- and how to recognise a boundary solution when you get one
6. **Extractant comparison** --- each inside its own validity window
7. **Provenance** --- asking any field where it came from

**Three things worth carrying out of here:**

- Counter-current is not cross-current. `(E**(N+1) - E)/(E**(N+1) - 1)`, not
  `1 - 1/(1+E)**N`; the second is roughly twice the first at the conditions in
  section 5.
- A converged-looking optimizer is not a converged optimizer. Section 7's
  ascent stops at a bound with a non-zero gradient, and says so.
- Only PC88A's pH coefficients are measured, and only five of the fifteen
  prices are sourced. Section 9 is how you find that out for any field.

**Next Steps:**
- See `21_custom_extractants.ipynb` to learn how to define custom extractants
- Explore pre-built flowsheet templates for complete separation trains
- Add economic analysis to evaluate process profitability
